# Kafka Demo — Lab 3

### Connect to Kafka Broker Server
Open an SSH tunnel in your terminal and leave it running while you use this notebook.
Replace `<NetID>` with your UIC NetID:
```
ssh -o ServerAliveInterval=60 -L 9092:localhost:9092 <NetID>@cs544-f26.cs.uic.edu -NTf
```

### To kill connection
```
lsof -ti:9092 | xargs kill -9
```

### Setup
```
python -m pip install kafka-python
```

See [bug_list.md](./bug_list.md) for frequent bugs and solutions.

In [2]:
import os
from datetime import datetime
from json import dumps, loads
from time import sleep
from random import randint
from kafka import KafkaConsumer, KafkaProducer

# Update this for your own recitation section :)
topic = 'recitation-3' # replace x with your recitation section

### Producer Mode -> Writes Data to Broker

In [8]:
# Create a producer to write data to kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaProducer.html

# [TODO]: Replace '...' with the address of your Kafka bootstrap server
producer = KafkaProducer(bootstrap_servers=['localhost:9092'],
                        value_serializer=lambda x: dumps(x).encode('utf-8'))

# [TODO]: Add cities of your choice
cities = ['Chicago', 'Cairo', 'Amman']

# Write data via the producer
print("Writing to Kafka Broker")
for i in range(10):
    data = f'{datetime.now().strftime("%Y-%m-%d %H:%M:%S")},{cities[randint(0,len(cities)-1)]},{randint(18, 32)}ºC'
    print(f"Writing: {data}")
    producer.send(topic=topic, value=data)
    sleep(1)

Writing to Kafka Broker
Writing: 2026-09-25 16:17:40,Cairo,30ºC
Writing: 2026-09-25 16:17:42,Amman,29ºC
Writing: 2026-09-25 16:17:43,Chicago,22ºC
Writing: 2026-09-25 16:17:44,Chicago,23ºC
Writing: 2026-09-25 16:17:45,Amman,26ºC
Writing: 2026-09-25 16:17:46,Cairo,19ºC
Writing: 2026-09-25 16:17:47,Cairo,23ºC
Writing: 2026-09-25 16:17:48,Chicago,26ºC
Writing: 2026-09-25 16:17:49,Amman,28ºC
Writing: 2026-09-25 16:17:50,Cairo,23ºC


### Consumer Mode -> Reads Data from Broker

In [ ]:
# Create a consumer to read data from kafka
# Ref: https://kafka-python.readthedocs.io/en/master/apidoc/KafkaConsumer.html

# [TODO]: Complete the missing ... parameters/arguments using the Kafka documentation
consumer = KafkaConsumer(topic,
    bootstrap_servers=['localhost:9092'],
    auto_offset_reset='earliest', #Experiment with different values
    # Commit that an offset has been read
    enable_auto_commit=True,
    # How often to tell Kafka, an offset has been read
    auto_commit_interval_ms=1000
)

print('Reading Kafka Broker')
for message in consumer:
    message = message.value.decode()
    # Default message.value type is bytes!
    print(loads(message))
    os.system(f"echo {message} >> kafka_log.csv")

Reading Kafka Broker
2026-09-25 15:56:40,Atlanta,26ºC
2026-09-25 15:56:42,Atlanta,27ºC
2026-09-25 15:56:43,Chicago,30ºC
2026-09-25 15:56:44,Cairo,32ºC
2026-09-25 15:56:45,Cairo,23ºC
2026-09-25 15:56:46,Chicago,25ºC
2026-09-25 15:56:47,Cairo,21ºC
2026-09-25 15:56:48,Cairo,24ºC
2026-09-25 15:56:49,Cairo,25ºC
2026-09-25 15:56:50,Atlanta,32ºC
2026-09-25 16:17:40,Cairo,30ºC
2026-09-25 16:17:42,Amman,29ºC
2026-09-25 16:17:43,Chicago,22ºC
2026-09-25 16:17:44,Chicago,23ºC
2026-09-25 16:17:45,Amman,26ºC
2026-09-25 16:17:46,Cairo,19ºC
2026-09-25 16:17:47,Cairo,23ºC
2026-09-25 16:17:48,Chicago,26ºC
2026-09-25 16:17:49,Amman,28ºC
2026-09-25 16:17:50,Cairo,23ºC


# Use kcat!
It's a CLI (Command Line Interface). Previously known as kafkacat


Ref: https://docs.confluent.io/platform/current/app-development/kafkacat-usage.html

In [ ]:
#kcat command: connect to local Kafka broker, specify a topic, and consume messages from the earliest offset